# Week 04 Lab 01: Attention, Logit Lens, and Saliency, with Causal Checks

**Scenario.** The Cordwell Home and Hardware ML platform team is preparing to defend its LLM stack in front of an internal model review board. Before anyone signs off on production use, the board wants proof that the team can explain, with evidence, why a model produced a given output. Your job in this lab is to build that evidence workflow on a small open model: form hypotheses about what attention heads are doing, watch a prediction form layer by layer, score which input tokens mattered, and then confirm or refute those scores with causal interventions.

**Duration:** about 120 to 150 minutes.

| Part | Topic | Time | You write code? |
|---|---|---|---|
| A | Setup, cache, attention heatmaps | 15-20 min | No (run and read) |
| B | Head fingerprinting and hypotheses | 20-25 min | Yes (2 functions) |
| C | Logit lens | 20-25 min | Yes (2 functions) |
| D | Saliency: Gradient x Input and Integrated Gradients | 25-30 min | Yes (2 functions) |
| E | Causal checks: ablation and activation patching | 25-30 min | Yes (2 functions) |
| F | Report | 10-15 min | Markdown only |
| Stretch | Attention x Gradient, token erasure, patching scan | optional | Optional |

**How this lab works.**

1. Cells marked `TODO` contain functions that raise `NotImplementedError`. Replace the `raise` line with your implementation. Everything else is pre-written plumbing so your time goes to the concepts, not the boilerplate.
2. After each task, a `check(...)` cell tells you `PASS`, `FAIL`, or `STUB`. Checks never crash the notebook, so Run All always completes.
3. Stuck? Open `HINTS.md`. It has three escalating levels per task. Level 2 should unstick you without giving the answer away.
4. Each part ends with short written questions. Answer them in the markdown cells provided. The report in Part F pulls it all together.


## Environment notes (read once)

- This lab needs `transformer_lens` 3.x, which pulls its own recent `transformers`. Use the fresh virtual environment described in `README.md` rather than your everyday course environment.
- The first run downloads the `gpt2` weights, roughly 500 MB. After that everything runs offline.
- No GPU is required. Everything here is sized for CPU or Apple MPS. The one slow cell is Integrated Gradients, which takes about a minute on CPU. That is normal, not broken.
- All results in this lab are deterministic: we never sample, we only read logits, activations, and gradients. The seed is set anyway as a habit.


In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from transformer_lens.model_bridge import TransformerBridge
from transformer_lens.utilities import get_act_name

torch.manual_seed(42)

def pick_device() -> str:
    """Auto-select the best available device: CUDA, then Apple MPS, then CPU."""
    try:
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"

DEVICE = pick_device()
MODEL_DIR = Path(os.environ.get("W4L1_MODEL_DIR", "~/models/gpt2")).expanduser().resolve()  # config variable, do not hard-code model paths
MODEL_NAME = str(MODEL_DIR)
print(f"Device: {DEVICE}   Model: {MODEL_NAME}")

## Loading the model through TransformerBridge

TransformerLens 3.x deprecates the old `HookedTransformer.from_pretrained(...)` entry point. The current path is `TransformerBridge.boot_transformers(...)`, which wraps a standard HuggingFace model and exposes the same hook system.

One important detail: by default the Bridge keeps the raw HuggingFace weights. The classic interpretability workflow (and most published results, including the papers this module cites) relies on preprocessed weights: LayerNorm folded into the surrounding linear layers and writing weights centered. `enable_compatibility_mode()` applies that preprocessing so our numbers line up with the legacy `HookedTransformer` behavior and with the literature.


In [ ]:
model = TransformerBridge.boot_transformers(MODEL_NAME, device=DEVICE)
model.enable_compatibility_mode(disable_warnings=True)

N_LAYERS = model.cfg.n_layers
N_HEADS = model.cfg.n_heads
print(f"Loaded {MODEL_NAME}: {N_LAYERS} layers, {N_HEADS} heads per layer, d_model={model.cfg.d_model}")

## The check helper

Run this once. Every task below has one or more `check(...)` calls. A check that hits a `NotImplementedError` reports `STUB` rather than `FAIL`, so a fresh Run All of this notebook shows 3 passes, 0 failures, and 16 stubs. Your goal is to turn the stubs into passes.


In [ ]:
CHECKS = {"pass": 0, "fail": 0, "stub": 0}

def check(name, fn):
    """Soft check: prints PASS, FAIL, or STUB. Never raises."""
    try:
        ok = bool(fn())
    except NotImplementedError:
        CHECKS["stub"] += 1
        print(f"[STUB] {name}: not implemented yet")
        return
    except Exception as e:
        CHECKS["fail"] += 1
        print(f"[FAIL] {name}: {type(e).__name__}: {e}")
        return
    if ok:
        CHECKS["pass"] += 1
        print(f"[PASS] {name}")
    else:
        CHECKS["fail"] += 1
        print(f"[FAIL] {name}: condition was False")

check("A setup: device is valid", lambda: DEVICE in ("cuda", "mps", "cpu"))
check("A setup: model booted with layers and heads", lambda: N_LAYERS >= 1 and N_HEADS >= 1)

## Part A: Cache the model's internals and look at attention (15-20 min, run and read)

**The idea.** `run_with_cache` runs a forward pass and records every intermediate activation into a dictionary. Two cache entries matter most today:

- `blocks.L.attn.hook_pattern`: the attention pattern for layer L, shape `[batch, head, query_pos, key_pos]`. Row q is a probability distribution over which earlier tokens position q reads from.
- `blocks.L.hook_resid_post`: the residual stream after layer L, shape `[batch, pos, d_model]`. This is the model's running working memory, and it is what the logit lens reads in Part C.

The prompts below are grouped by what they are good at exposing. Repeated-pattern prompts light up previous-token and induction behavior. The factual prompts give the logit lens a clean single-answer target.


In [ ]:
PROMPTS_PATTERN = [
    "alpha beta alpha beta alpha beta alpha beta alpha",
    "foo bar foo bar foo bar foo bar foo",
    "red green red green red green red green red",
]

PROMPTS_AGREEMENT = [
    "Sarah handed the keys to John because she",
    "When the doctor met the patient, he",
    "The violinist thanked the conductor after she",
]

PROMPTS_FACTUAL = [
    "The capital of Japan is",
    "The Eiffel Tower is located in",
    "Python was created by",
]

prompt_a = PROMPTS_PATTERN[0]
tokens_a = model.to_tokens(prompt_a)          # shape [1, seq_len], BOS prepended
str_tokens_a = model.to_str_tokens(prompt_a)  # list of readable token strings

print("Token count:", tokens_a.shape[1])
print("Tokens:", str_tokens_a)

with torch.no_grad():
    logits_a, cache_a = model.run_with_cache(tokens_a)

print("Logits shape:", tuple(logits_a.shape))
print("Example cache keys:", [k for k in list(cache_a.keys()) if "blocks.0." in k][:6])

Note the first token: `to_tokens` prepends a beginning-of-sequence token, which for GPT-2 is the string `<|endoftext|>`. Every position-indexed result in this lab (attention rows, saliency scores) includes that BOS position at index 0. GPT-2 heads often park attention on BOS when they have nothing better to do; treat it as a resting position, not as evidence that BOS is meaningful.


In [ ]:
def plot_attention(cache, layer, head, str_tokens, extra=""):
    """Heatmap of one head's attention pattern. Rows are query positions, columns are key positions."""
    pattern = cache[get_act_name("pattern", layer)][0, head].detach().cpu().numpy()
    n = len(str_tokens)
    fig, ax = plt.subplots(figsize=(0.55 * n + 2, 0.55 * n + 1.5))
    im = ax.imshow(pattern, cmap="Blues", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(n)); ax.set_xticklabels(str_tokens, rotation=90)
    ax.set_yticks(range(n)); ax.set_yticklabels(str_tokens)
    ax.set_xlabel("key position (attended to)")
    ax.set_ylabel("query position (attending from)")
    ax.set_title(f"L{layer} H{head} {extra}".strip())
    fig.colorbar(im, fraction=0.03)
    plt.tight_layout()
    plt.show()

# Worked example: one head from an early layer on the repeated pattern prompt.
plot_attention(cache_a, layer=min(4, N_LAYERS - 1), head=min(11, N_HEADS - 1), str_tokens=str_tokens_a)

check("A cache: attention rows are probability distributions",
      lambda: torch.allclose(cache_a[get_act_name("pattern", 0)][0].sum(dim=-1),
                             torch.ones(N_HEADS, tokens_a.shape[1], device=DEVICE), atol=1e-3))

**Worked target output for Part A.** With `gpt2` you should see a heatmap where most attention weight forms a visible structure rather than a uniform smear. On the repeated `alpha beta ...` prompt, the head plotted above (layer 4, head 11) is widely reported in the interpretability literature as a previous-token head, so expect a bright band one step below the diagonal: each token reading the token right before it. If your plot looks different, that is fine, it is data, not an error. Confirm what you actually see and note it.

**Question A.1** (answer in the next cell): pick any one head that shows visible structure. In one sentence, state a testable hypothesis about what it tracks. Example shape: "Layer L head H attends from each token to the previous occurrence of the same word."


*Your answer to A.1 here.*


## Part B: Fingerprint heads with a score, then hypothesize (20-25 min)

Eyeballing heatmaps does not scale to 144 heads. The standard move is to reduce each head's pattern to a single number that measures one specific behavior, then scan every head.

**Task B1.** Implement `prev_token_score`. Given one prompt's attention pattern for a whole layer, return a per-head score: the average attention weight each query position q puts on key position q - 1. A perfect previous-token head scores 1.0. A head that ignores the previous token scores near 0.

**Worked target output.** For a synthetic 4-token pattern where head 0 attends entirely to the previous token and head 1 attends only to itself, the function must return exactly `[1.0, 0.0]`. The first check below tests exactly that, so you can develop against it before touching real model data.


In [ ]:
def prev_token_score(pattern: torch.Tensor) -> torch.Tensor:
    """Score how strongly each head attends to the immediately preceding token.

    Args:
        pattern: attention pattern for one prompt, shape [n_heads, n_query, n_key],
                 where pattern[h, q, k] is head h's attention from query q to key k.

    Returns:
        Tensor of shape [n_heads]. For each head, the mean over query positions
        q = 1 .. n_query - 1 of pattern[h, q, q - 1].
    """
    # TODO (Task B1)
    raise NotImplementedError("Task B1")

In [ ]:
def _b1_test_pattern():
    p = torch.zeros(2, 4, 4)
    p[0, range(1, 4), range(0, 3)] = 1.0  # head 0: pure previous-token
    p[1] = torch.eye(4)                   # head 1: pure self-attention
    return p

check("B1 shape: returns one score per head",
      lambda: prev_token_score(_b1_test_pattern()).shape == (2,))
check("B1 values: perfect prev-token head scores 1.0, self head scores 0.0",
      lambda: torch.allclose(prev_token_score(_b1_test_pattern()), torch.tensor([1.0, 0.0]), atol=1e-6))

**Task B2.** Implement `scan_prev_token`. Loop over every layer, pull that layer's pattern out of the cache with `get_act_name("pattern", layer)`, score it with your B1 function, and stack the results into a `[n_layers, n_heads]` tensor. The plotting cell after the checks turns it into a heatmap of the whole model.


In [ ]:
def scan_prev_token(cache) -> torch.Tensor:
    """Score every head in the model for previous-token behavior.

    Args:
        cache: an ActivationCache from model.run_with_cache on a single prompt.

    Returns:
        Tensor of shape [n_layers, n_heads] of prev_token_score values.
        Use cache[get_act_name("pattern", layer)][0] to get one layer's pattern.
    """
    # TODO (Task B2)
    raise NotImplementedError("Task B2")

In [ ]:
check("B2 shape: one score per (layer, head)",
      lambda: scan_prev_token(cache_a).shape == (N_LAYERS, N_HEADS))
check("B2 range: attention-derived scores stay in [0, 1]",
      lambda: (lambda s: bool((s >= -1e-6).all() and (s <= 1 + 1e-6).all()))(scan_prev_token(cache_a)))

In [ ]:
try:
    scores_b = scan_prev_token(cache_a)
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(scores_b.detach().cpu().numpy(), cmap="viridis", aspect="auto")
    ax.set_xlabel("head"); ax.set_ylabel("layer")
    ax.set_title("Previous-token score, every head")
    fig.colorbar(im, fraction=0.03)
    plt.tight_layout(); plt.show()
    top = torch.topk(scores_b.flatten(), 5)
    for v, i in zip(top.values, top.indices):
        print(f"L{i // N_HEADS} H{i % N_HEADS}: {v:.3f}")
except NotImplementedError:
    print("Implement Tasks B1 and B2 to see the full-model scan.")

**Worked target output for B2.** A `[12, 12]` heatmap with most cells dark (near the uniform baseline, roughly 1 over the sequence length) and a small number of bright outliers. With `gpt2` on this prompt, expect one or two heads scoring above 0.8. The literature identifies layer 4 head 11 and layer 2 head 2 as previous-token heads in this model, so those are the outliers you should expect near the top of the printed list.

**Question B.1.** For your top two heads, write a one-sentence hypothesis each.

**Question B.2.** Build a small evidence table for one head, following this worked example format:

| prompt | query token (pos) | most-attended key token (pos) | weight |
|---|---|---|---|
| alpha beta alpha ... | beta (2) | alpha (1) | 0.91 |
| alpha beta alpha ... | alpha (3) | beta (2) | 0.88 |

**Question B.3.** For each hypothesis, name at least one counter-example that would falsify it. A previous-token hypothesis, for instance, is falsified by a prompt where the head instead tracks the previous occurrence of the same word rather than the adjacent token.


*Your answers to B.1, B.2, B.3 here.*


## Part C: Logit lens (20-25 min)

**The idea.** The residual stream after each layer can be decoded into vocabulary space at any depth, not just at the end: apply the final LayerNorm, then multiply by the unembedding matrix. Reading the top predictions at each layer shows the model's belief forming. This is the logit lens.

The recipe per layer, at the final token position:

1. `resid = cache[get_act_name("resid_post", layer)][0, -1]` gives the `[d_model]` residual vector.
2. `model.ln_final(resid)` applies the final LayerNorm.
3. `@ model.W_U + model.b_U` maps it to `[vocab]` logits.

A useful correctness property, and the second check below: at the last layer this reproduces the model's actual output exactly, because that is literally how the model computes its logits.

**Task C1.** Implement `logit_lens_topk`: for each layer, return the top-k predicted token strings and their logits at the final position.

**Worked target output.** The printing cell renders a table like this (values here are illustrative):

| layer | top1 | top2 | top3 |
|---|---|---|---|
| 0 | ` the` | ` a` | ` in` |
| ... | | | |
| 11 | ` Tokyo` | ` Japan` | ` the` |

With `gpt2` on `The capital of Japan is`, the correct city typically only becomes the top prediction in the last few layers. Early layers produce generic filler. Watching where the answer first appears is the whole point.


In [ ]:
def logit_lens_topk(cache, k: int = 5) -> list:
    """Decode the residual stream after every layer into top-k predictions.

    Args:
        cache: ActivationCache from run_with_cache on a single prompt.
        k: number of top predictions to keep per layer.

    Returns:
        A list with one entry per layer: (layer_index, token_strings, logit_values)
        where token_strings is a list of k decoded strings for the final position
        and logit_values is the matching list of k floats, both sorted best first.
        Decode a token id with model.tokenizer.decode([token_id]).
    """
    # TODO (Task C1)
    raise NotImplementedError("Task C1")

In [ ]:
prompt_c = PROMPTS_FACTUAL[0]
tokens_c = model.to_tokens(prompt_c)
with torch.no_grad():
    logits_c, cache_c = model.run_with_cache(tokens_c)
TARGET_C = int(logits_c[0, -1].argmax())
print(f"Prompt: {prompt_c!r}   model's top next token: {model.tokenizer.decode([TARGET_C])!r}")

check("C1 length: one row per layer",
      lambda: len(logit_lens_topk(cache_c, k=3)) == N_LAYERS)
check("C1 exactness: final layer's top1 equals the model's actual prediction",
      lambda: logit_lens_topk(cache_c, k=1)[-1][1][0] == model.tokenizer.decode([TARGET_C]))

try:
    for layer, strings, values in logit_lens_topk(cache_c, k=5):
        pretty = "  ".join(f"{s!r}:{v:.2f}" for s, v in zip(strings, values))
        print(f"layer {layer:2d}  {pretty}")
except NotImplementedError:
    print("Implement Task C1 to see the layer-by-layer table.")

**Task C2.** Implement `target_logit_by_layer`: the logit of one specific target token, read through the lens at every layer. This turns the table above into a single curve: how strongly does each depth of the network believe in the final answer?


In [ ]:
def target_logit_by_layer(cache, target_id: int) -> torch.Tensor:
    """Track one token's logit through every layer of the logit lens.

    Args:
        cache: ActivationCache from run_with_cache on a single prompt.
        target_id: the vocabulary id of the token to track.

    Returns:
        Tensor of shape [n_layers]: the lens logit of target_id at the final
        position after each layer. The last value must equal the model's real
        output logit for that token.
    """
    # TODO (Task C2)
    raise NotImplementedError("Task C2")

In [ ]:
check("C2 length: one value per layer",
      lambda: target_logit_by_layer(cache_c, TARGET_C).shape == (N_LAYERS,))
check("C2 exactness: last layer matches the model's real logit",
      lambda: torch.isclose(target_logit_by_layer(cache_c, TARGET_C)[-1],
                            logits_c[0, -1, TARGET_C].cpu(), atol=1e-2))

try:
    curve = target_logit_by_layer(cache_c, TARGET_C)
    plt.figure(figsize=(7, 3.5))
    plt.plot(range(N_LAYERS), curve.numpy(), marker="o")
    plt.xlabel("layer"); plt.ylabel(f"lens logit of {model.tokenizer.decode([TARGET_C])!r}")
    plt.title(f"Belief in the answer, layer by layer: {prompt_c!r}")
    plt.tight_layout(); plt.show()
except NotImplementedError:
    print("Implement Task C2 to see the belief curve.")

**Question C.1.** In 4 to 6 sentences, narrate how the model's belief evolved on your factual prompt: where does the answer first enter the top 5, where does it take over top 1, and does the curve rise smoothly or jump?

**Question C.2.** Did any early layer strongly favor a different completion than the final answer? Show the layer and tokens as evidence.

**A caution before you over-trust this.** The logit lens reads intermediate states with the final layer's decoder. Early layers were never trained to be readable that way, so a garbled early reading means "not yet decodable," not necessarily "the model believes nothing." Treat the lens as descriptive evidence, to be confirmed causally in Part E.


*Your answers to C.1 and C.2 here.*


## Part D: Saliency, two ways (25-30 min)

**The idea.** Saliency methods score how much each input token mattered for a chosen output. You will implement two gradient-based methods and compare them:

- **Gradient x Input**: one backward pass. Multiply each token's embedding by the gradient of the target logit with respect to that embedding, and sum over the embedding dimension. Fast, but it is a local linear approximation and can be noisy.
- **Integrated Gradients (IG)**: average that gradient along a straight path from a zero baseline to the real embeddings, then multiply by the input difference. Slower (one backward pass per step) but it satisfies a built-in accounting identity called completeness: the scores must sum to the difference between the model's output on the real input and on the baseline. Our check exploits that identity, which means your implementation gets verified against mathematics rather than against a hidden expected value.

To take gradients with respect to embeddings, we run the model while substituting our own embedding tensor via the `hook_embed` hook. That helper is pre-written below because the hook plumbing is not the lesson; the gradient logic is.


In [ ]:
def forward_with_embeds(tokens, embeds):
    """Run the model on tokens, but replace the embedding activations with embeds.

    embeds must have shape [1, seq_len, d_model]. Gradients flow back into embeds.
    """
    def replace(act, hook):
        return embeds
    with model.hooks(fwd_hooks=[("hook_embed", replace)]):
        return model(tokens)

def plot_token_bars(str_tokens, scores, title):
    """Bar chart of one score per token. Lengths must match, including the BOS position."""
    scores = scores.detach().cpu().numpy()
    assert len(str_tokens) == len(scores), (
        f"{len(str_tokens)} token strings vs {len(scores)} scores; "
        "every position, including BOS at index 0, gets exactly one score")
    plt.figure(figsize=(8, 3))
    plt.bar(range(len(str_tokens)), scores)
    plt.xticks(range(len(str_tokens)), str_tokens, rotation=90)
    plt.title(title)
    plt.tight_layout(); plt.show()

**Task D1.** Implement `gradient_x_input`. Recipe:

1. Get the real embeddings: `model.embed(tokens)`, then `.detach().clone().requires_grad_(True)` so they become a leaf tensor you can differentiate.
2. Run `forward_with_embeds(tokens, embeds)` and take the logit at position `[0, -1, target_id]`.
3. Call `.backward()` on that scalar.
4. Return `(embeds.grad * embeds).sum(dim=-1)[0].detach()`, one score per token position.

**Worked target output.** A tensor of shape `[seq_len]` (including BOS), finite everywhere, plotted by the cell after the checks as a labeled bar chart, positive bars pushing the target up and negative bars pushing it down.


In [ ]:
def gradient_x_input(tokens, target_id: int) -> torch.Tensor:
    """Gradient x Input saliency for the final position's target_id logit.

    Args:
        tokens: [1, seq_len] token ids.
        target_id: vocabulary id whose logit we explain.

    Returns:
        Tensor of shape [seq_len], one score per input position (BOS included).
    """
    # TODO (Task D1)
    raise NotImplementedError("Task D1")

In [ ]:
prompt_d = PROMPTS_AGREEMENT[0]
tokens_d = model.to_tokens(prompt_d)
str_tokens_d = model.to_str_tokens(prompt_d)
with torch.no_grad():
    TARGET_D = int(model(tokens_d)[0, -1].argmax())
print(f"Prompt: {prompt_d!r}   target token: {model.tokenizer.decode([TARGET_D])!r}")

check("D1 shape: one score per input position",
      lambda: gradient_x_input(tokens_d, TARGET_D).shape == (tokens_d.shape[1],))
check("D1 sanity: scores are finite and not all zero",
      lambda: (lambda s: bool(torch.isfinite(s).all() and s.abs().sum() > 0))(
          gradient_x_input(tokens_d, TARGET_D)))

try:
    plot_token_bars(str_tokens_d, gradient_x_input(tokens_d, TARGET_D),
                    f"Gradient x Input: {prompt_d!r}")
except NotImplementedError:
    print("Implement Task D1 to see the saliency bars.")

**Task D2.** Implement `integrated_gradients`. Recipe:

1. `actual = model.embed(tokens).detach()` and `baseline = torch.zeros_like(actual)`.
2. Loop `i` from 0 to `steps - 1` using the midpoint rule: `alpha = (i + 0.5) / steps`. At each step build `interp = baseline + alpha * (actual - baseline)`, detached, cloned, with `requires_grad_(True)`; run `forward_with_embeds`; backprop the `[0, -1, target_id]` logit; accumulate `interp.grad`.
3. Average the accumulated gradients over steps, multiply elementwise by `(actual - baseline)`, sum over the embedding dimension, and return shape `[seq_len]`.

The midpoint rule matters: it makes the completeness identity hold to numerical precision, which is exactly what the second check verifies. This cell is the slow one, about a minute at 64 steps on CPU.


In [ ]:
def integrated_gradients(tokens, target_id: int, steps: int = 64) -> torch.Tensor:
    """Integrated Gradients from a zero-embedding baseline to the real embeddings.

    Args:
        tokens: [1, seq_len] token ids.
        target_id: vocabulary id whose logit we explain.
        steps: number of interpolation steps (midpoint rule: alpha = (i + 0.5) / steps).

    Returns:
        Tensor of shape [seq_len]. Completeness must hold: the scores sum to
        f(actual) - f(baseline), where f is the target logit at the final position.
    """
    # TODO (Task D2)
    raise NotImplementedError("Task D2")

In [ ]:
def _ig_completeness_ok(tol=0.05, steps=128):
    # The zero-embedding baseline sits in a highly nonlinear region (LayerNorm
    # near zero norm), so the straight-line integrand is steep and the midpoint
    # sum needs many steps to converge. 32 steps is far from converged here
    # (rel err ~1.8); 128 brings it comfortably under tol.
    ig = integrated_gradients(tokens_d, TARGET_D, steps=steps)
    with torch.no_grad():
        f_actual = model(tokens_d)[0, -1, TARGET_D]
        f_baseline = forward_with_embeds(
            tokens_d, torch.zeros_like(model.embed(tokens_d)))[0, -1, TARGET_D]
    gap = float(f_actual - f_baseline)
    rel_err = abs(float(ig.sum()) - gap) / (abs(gap) + 1e-8)
    print(f"    IG sum {float(ig.sum()):.4f}   f(actual) - f(baseline) {gap:.4f}   rel err {rel_err:.4f}")
    return rel_err < tol

check("D2 shape: one score per input position",
      lambda: integrated_gradients(tokens_d, TARGET_D, steps=8).shape == (tokens_d.shape[1],))
check("D2 completeness: scores sum to f(actual) minus f(baseline)", _ig_completeness_ok)

try:
    ig_d = integrated_gradients(tokens_d, TARGET_D, steps=64)
    plot_token_bars(str_tokens_d, ig_d, f"Integrated Gradients: {prompt_d!r}")
except NotImplementedError:
    print("Implement Task D2 to see the IG bars.")

In [ ]:
# Side-by-side comparison on a second prompt (factual). Runs once D1 and D2 exist.
try:
    tokens_f = model.to_tokens(PROMPTS_FACTUAL[0])
    str_tokens_f = model.to_str_tokens(PROMPTS_FACTUAL[0])
    with torch.no_grad():
        tgt_f = int(model(tokens_f)[0, -1].argmax())
    plot_token_bars(str_tokens_f, gradient_x_input(tokens_f, tgt_f),
                    f"Gradient x Input: {PROMPTS_FACTUAL[0]!r}")
    plot_token_bars(str_tokens_f, integrated_gradients(tokens_f, tgt_f, steps=64),
                    f"Integrated Gradients: {PROMPTS_FACTUAL[0]!r}")
except NotImplementedError:
    print("Implement Tasks D1 and D2 to compare methods across prompts.")

**Question D.1.** Where do the two methods agree and where do they disagree? Give 2 or 3 concrete observations naming tokens.

**Question D.2.** Which method felt more stable when you reran it on the sibling prompts in the same group? Why would you expect that from how each is computed?

**A caution.** High saliency is a correlation-flavored claim, not proof of causation, and attention weights in particular are known to be an unreliable explanation on their own. Part E is where claims earn their causal keep.


*Your answers to D.1 and D.2 here.*


## Part E: Causal checks, ablation and activation patching (25-30 min)

**The idea.** Descriptive tools (heatmaps, lenses, saliency) tell you where to look. Causal tools tell you whether the story is true, by editing the computation and measuring the damage:

- **Head ablation**: zero out one head's output and measure how much the target's log probability drops. If your "important" head can be deleted for free, the story was wrong.
- **Activation patching**: run a clean prompt and a corrupted prompt that differ in one key detail, then splice one head's clean activations into the corrupted run. If that single splice moves the output back toward the clean answer, that head causally carries the detail.

The prompt pair below is the classic indirect-object-identification setup from the interpretability literature: the two prompts differ only in which name repeats, which flips the correct completion between the two names. The assert guarantees both tokenize to the same length so activations line up position by position.


In [ ]:
CLEAN_PROMPT = "When John and Mary went to the store, John gave a drink to"
CORRUPT_PROMPT = "When John and Mary went to the store, Mary gave a drink to"

clean_tokens = model.to_tokens(CLEAN_PROMPT)
corrupt_tokens = model.to_tokens(CORRUPT_PROMPT)
assert clean_tokens.shape == corrupt_tokens.shape, "prompt pair must tokenize to equal lengths"

with torch.no_grad():
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)
    corrupt_logits = model(corrupt_tokens)

TARGET_E = int(clean_logits[0, -1].argmax())
CLEAN_LOGIT = float(clean_logits[0, -1, TARGET_E])
CORRUPT_LOGIT = float(corrupt_logits[0, -1, TARGET_E])
print(f"Clean answer token: {model.tokenizer.decode([TARGET_E])!r}")
print(f"Its logit on the clean prompt: {CLEAN_LOGIT:.3f}   on the corrupted prompt: {CORRUPT_LOGIT:.3f}")

With `gpt2`, the clean answer should be the name that did not repeat (expect `' Mary'`), and its logit should be visibly lower on the corrupted prompt. That gap is the recovery yardstick for patching.

**Task E1.** Implement `ablate_head`. Zero out head `head` in layer `layer` by hooking `get_act_name("z", layer)` (per-head outputs, shape `[batch, pos, head, d_head]`), and report the target's log probability before and after. Use `torch.log_softmax` over the final position's logits. Inside the hook, clone `z` before editing it.

**Worked target output.** A dict like `{"base_logprob": -1.92, "ablated_logprob": -2.31, "delta_logprob": -0.39}` (illustrative numbers). A negative delta means the head was helping the target.


In [ ]:
def ablate_head(tokens, layer: int, head: int, target_id: int) -> dict:
    """Zero one attention head and measure the effect on the target token.

    Args:
        tokens: [1, seq_len] token ids.
        layer, head: which head to zero out, via the hook get_act_name("z", layer).
                     z has shape [batch, pos, head, d_head]; set the head slice to 0.
        target_id: vocabulary id to score at the final position.

    Returns:
        {"base_logprob": float, "ablated_logprob": float, "delta_logprob": float}
        where delta_logprob = ablated_logprob - base_logprob. Run under torch.no_grad().
    """
    # TODO (Task E1)
    raise NotImplementedError("Task E1")

In [ ]:
_E_LAYER = max(0, N_LAYERS - 3)      # layer 9 in gpt2
_E_HEAD = min(9, N_HEADS - 1)        # head 9 in gpt2: a reported name-mover head

check("E1 contract: returns the three logprob fields",
      lambda: set(ablate_head(clean_tokens, _E_LAYER, _E_HEAD, TARGET_E)) ==
              {"base_logprob", "ablated_logprob", "delta_logprob"})
check("E1 effect: ablating a head actually changes the output",
      lambda: abs(ablate_head(clean_tokens, _E_LAYER, _E_HEAD, TARGET_E)["delta_logprob"]) > 1e-9)

try:
    r = ablate_head(clean_tokens, _E_LAYER, _E_HEAD, TARGET_E)
    print(f"Ablating L{_E_LAYER} H{_E_HEAD}: " +
          "  ".join(f"{k} {v:.4f}" for k, v in r.items()))
except NotImplementedError:
    print("Implement Task E1 to run the ablation.")

**Task E2.** Implement `patch_head_z`. Run the corrupted prompt, but hook `get_act_name("z", layer)` and overwrite head `head`'s slice with the clean run's cached values (`clean_cache[get_act_name("z", layer)]`). Report the patched logit of the target and the recovery fraction:

`recovery = (patched_logit - corrupt_logit) / (clean_logit - corrupt_logit)`

Recovery near 1 means this one head restored the clean behavior almost entirely. Near 0 means it carried nothing. Values outside [0, 1] happen and are worth remarking on.

**Worked target output.** A dict like `{"patched_logit": 14.1, "recovery": 0.62}` (illustrative numbers).


In [ ]:
def patch_head_z(corrupt_tokens, layer: int, head: int, clean_cache,
                 target_id: int, clean_logit: float, corrupt_logit: float) -> dict:
    """Patch one head's clean activations into the corrupted run.

    Args:
        corrupt_tokens: [1, seq_len] ids of the corrupted prompt.
        layer, head: which head to patch, via get_act_name("z", layer).
        clean_cache: ActivationCache from the clean prompt (same length).
        target_id: vocabulary id to score at the final position.
        clean_logit, corrupt_logit: the target's unpatched logits on each prompt.

    Returns:
        {"patched_logit": float, "recovery": float} with
        recovery = (patched_logit - corrupt_logit) / (clean_logit - corrupt_logit).
        Run under torch.no_grad(); clone z inside the hook before editing.
    """
    # TODO (Task E2)
    raise NotImplementedError("Task E2")

In [ ]:
check("E2 contract: returns patched_logit and recovery",
      lambda: set(patch_head_z(corrupt_tokens, _E_LAYER, _E_HEAD, clean_cache,
                               TARGET_E, CLEAN_LOGIT, CORRUPT_LOGIT)) ==
              {"patched_logit", "recovery"})
check("E2 effect: patching a head changes the corrupted run's output",
      lambda: abs(patch_head_z(corrupt_tokens, _E_LAYER, _E_HEAD, clean_cache,
                               TARGET_E, CLEAN_LOGIT, CORRUPT_LOGIT)["patched_logit"]
                  - CORRUPT_LOGIT) > 1e-9)

try:
    print(f"{'layer':>5} {'head':>4} {'ablate dLogP':>13} {'patch recovery':>15}")
    for layer, head in [(_E_LAYER, _E_HEAD), (max(0, N_LAYERS - 3), min(6, N_HEADS - 1)),
                        (min(4, N_LAYERS - 1), min(11, N_HEADS - 1)), (0, 0)]:
        a = ablate_head(clean_tokens, layer, head, TARGET_E)["delta_logprob"]
        p = patch_head_z(corrupt_tokens, layer, head, clean_cache,
                         TARGET_E, CLEAN_LOGIT, CORRUPT_LOGIT)["recovery"]
        print(f"{layer:>5} {head:>4} {a:>13.4f} {p:>15.4f}")
except NotImplementedError:
    print("Implement Tasks E1 and E2 to run the comparison table.")

**Question E.1.** Which head in your table shows real causal weight, by both measures? Give the numbers and a 2 or 3 sentence rationale. With `gpt2`, layer 9 head 9 is reported in the literature as a name-mover head for exactly this task, so it is the one to watch.

**Question E.2.** Compare against your Part D saliency results. Did any token or head look important descriptively but carry little causal weight, or the reverse? One example each way if you can find them.


*Your answers to E.1 and E.2 here.*


## Part F: Report (10-15 min)

Fill in the template below in this markdown cell. This is the artifact you would hand the review board.

**Method details.** Which techniques you ran, with parameters (IG steps, which layers and heads, how targets were chosen).

**Experimental setup.** Model name, device, prompts used, and the fact that everything is deterministic (no sampling).

**Findings.** Your strongest head hypothesis with its descriptive and causal evidence, and your logit lens narrative.

**Validation and uncertainty.** At least two limitations. Candidates: the logit lens misreads early layers, attention is not explanation, a zero baseline for IG is a choice not a truth, single prompts do not generalize, head roles reported for gpt2 do not transfer to other models.

**Reproducibility checklist.** Package versions (`transformer_lens`, `torch`), model id, device, seed, prompt strings, and IG step count. Enough for a classmate to reproduce your numbers exactly.


*Your report here.*


## Stretch goals (optional)

Pick any of these if you finish early. No checks; solutions are in the instructor's solution notebook.

**Stretch 1: Attention x Gradient.** A third saliency method: capture one layer's attention pattern during the forward pass (use `retain_grad()` on the activation inside a hook), backprop the target logit, multiply pattern by its gradient, and sum over heads and query positions to get one score per key token. Compare it to D1 and D2.

**Stretch 2: Token erasure.** Zero out one token's embedding (via `forward_with_embeds`) and measure the drop in the target's log probability. Erase the top token flagged by each Part D method and see whether the causal effect matches the saliency ranking.

**Stretch 3: Patching scan.** Run `patch_head_z` for every layer and head, and render the recovery fractions as a `[n_layers, n_heads]` heatmap. This reproduces the signature figure of the activation patching literature; the bright cells are your circuit.


## Final tally

Rerun this after finishing your tasks. Target: 19 passes, 0 failures, 0 stubs.


In [ ]:
print(f"PASS {CHECKS['pass']}   FAIL {CHECKS['fail']}   STUB {CHECKS['stub']}")